# 18 — Exactly-Once / Deduplication (Amazon FAR style)

At-least-once delivery means duplicates; consumers dedupe to get **effectively-once** processing.
Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch tasks (bounded-memory dedup,
then sharded parallel counting). Fill stubs, run each test cell, peek at the collapsed `ref_` solutions
only after trying. Events are identified by an `event_id`.

---

## Part 1 — Basic deduper

`Deduper` with `process(event_id) -> bool`: `True` the first time an id is seen (process it), `False`
on any repeat (drop it).

**Lock down:** a set of seen ids; this is the consumer-side idempotency that makes at-least-once
delivery safe.

In [ ]:
class Deduper:
    def __init__(self):
        raise NotImplementedError

    def process(self, event_id):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    d = Deduper()
    assert d.process("a") is True
    assert d.process("a") is False
    assert d.process("b") is True
    assert d.process("b") is False
    print("Part 1 OK")

_t1()

## Part 2 — TTL deduper

Remembering every id forever is unbounded. `TTLDeduper(window)` with `process(event_id, now) -> bool`
remembers an id only for `window` time: a repeat **within** the window is a duplicate (`False`); once
the window elapses the id is forgotten and a new sighting is processed again (`True`). Clock is
injected via `now`.

**Lock down:** a (re)sighting (re)starts the window; this bounds memory by how many *recent* ids you've
seen. Trade-off: a very-late duplicate (past the window) slips through.

In [ ]:
class TTLDeduper:
    def __init__(self, window):
        raise NotImplementedError

    def process(self, event_id, now):
        raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    d = TTLDeduper(window=10)
    assert d.process("a", now=0) is True
    assert d.process("a", now=5) is False              # within window
    assert d.process("a", now=100) is True             # window elapsed -> forgotten
    assert d.process("a", now=105) is False            # within new window
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe deduper (exactly-once)

`ConcurrentDeduper`: thread-safe `process(event_id)`. With many producers delivering an overlapping set
of ids, **each id must be processed exactly once in total**.

**The invariant:** 8 threads each call `process` on ids `e0..e99`; exactly **100** calls return `True`.
**Discuss:** the check-then-add race (two threads both seeing "not seen" → double processing) and why
it must be atomic; sharding the lock by id hash for throughput.

In [ ]:
import threading


class ConcurrentDeduper:
    def __init__(self):
        raise NotImplementedError

    def process(self, event_id):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    d = ConcurrentDeduper()
    trues, lock = [], threading.Lock()

    def worker():
        local = sum(1 for i in range(100) if d.process("e%d" % i))
        with lock:
            trues.append(local)

    ts = [threading.Thread(target=worker) for _ in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert sum(trues) == 100                            # 100 distinct ids, processed once total
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Bounded-memory deduper

`LRUDeduper(capacity)`: remember only the most recent `capacity` distinct ids (LRU eviction). A repeat
of a still-remembered id returns `False` (and refreshes it); an id that has been **evicted** is treated
as new again (`True`).

**Discuss:** the memory/accuracy trade-off (evicted ids cause missed-duplicate "leaks"), and that this
is the practical shape of dedup at scale — a finite cache, not an infinite set. The recurring
error-tolerance theme.

In [ ]:
from collections import OrderedDict


class LRUDeduper:
    def __init__(self, capacity):
        raise NotImplementedError

    def process(self, event_id):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    d = LRUDeduper(capacity=2)
    assert d.process("a") is True
    assert d.process("b") is True
    assert d.process("a") is False                      # still remembered (refreshes it)
    assert d.process("c") is True                       # evicts LRU "b"
    assert d.process("b") is True                       # "b" was evicted -> treated as new
    assert d.process("a") is True                       # adding "b" evicted "a" -> new again
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Sharded parallel distinct count

`parallel_distinct(events, num_shards, num_procs) -> int`: count the number of distinct event ids
across a huge list. **Shard by id** (`hash(id) % num_shards`) so each id lands in exactly one shard,
count distinct per shard in parallel with `ProcessPoolExecutor`, then sum. Worker is
`dedup_workers.count_distinct`.

**Discuss:** why sharding *by id* makes per-shard distinct counts additive (no id spans shards); GIL
(set work is CPU-bound -> processes); and that a global `set()` doesn't scale (memory + one lock).

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import dedup_workers


def parallel_distinct(events, num_shards, num_procs) -> int:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    events = ["e%d" % (i % 50) for i in range(500)]     # 50 distinct ids, each seen 10x
    assert parallel_distinct(events, num_shards=4, num_procs=2) == 50
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Probabilistic dedup at scale: Bloom filter (false positives = dropped non-duplicates) vs exact.
- Idempotency keys end-to-end; dedup windows tied to delivery retry bounds.
- Persistent/distributed dedup store; partition by key + TTL eviction.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from collections import OrderedDict
from concurrent.futures import ProcessPoolExecutor
import dedup_workers


class RefDeduper:
    def __init__(self):
        self.seen = set()

    def process(self, event_id):
        if event_id in self.seen:
            return False
        self.seen.add(event_id)
        return True


class RefTTLDeduper:
    def __init__(self, window):
        self.window = window
        self.seen = {}                                  # id -> time window started

    def process(self, event_id, now):
        first = self.seen.get(event_id)
        if first is not None and now - first < self.window:
            return False
        self.seen[event_id] = now                       # (re)start the window
        return True


class RefConcurrentDeduper:
    def __init__(self):
        self.seen = set()
        self.lock = threading.Lock()

    def process(self, event_id):
        with self.lock:                                 # atomic check-then-add
            if event_id in self.seen:
                return False
            self.seen.add(event_id)
            return True


class RefLRUDeduper:
    def __init__(self, capacity):
        self.cap = capacity
        self.seen = OrderedDict()

    def process(self, event_id):
        if event_id in self.seen:
            self.seen.move_to_end(event_id)
            return False
        self.seen[event_id] = True
        if len(self.seen) > self.cap:
            self.seen.popitem(last=False)               # evict least-recently-used
        return True


def ref_parallel_distinct(events, num_shards, num_procs):
    shards = [[] for _ in range(num_shards)]
    for e in events:
        shards[hash(e) % num_shards].append(e)          # same id -> same shard
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return sum(ex.map(dedup_workers.count_distinct, shards))


_d = RefDeduper(); assert _d.process("a") and not _d.process("a")
_t = RefTTLDeduper(10); assert _t.process("a", 0) and not _t.process("a", 3) and _t.process("a", 20)
_l = RefLRUDeduper(1); assert _l.process("a") and _l.process("b") and _l.process("a")  # cap 1: a evicted by b
_c = RefConcurrentDeduper(); assert _c.process("x") and not _c.process("x")
assert ref_parallel_distinct(["a", "a", "b", "c", "c"], 3, 2) == 3
print("reference solutions OK")